In [1]:
from pathlib import Path
from typing import Union
import geopandas as gpd
import folium

In [2]:
from pathlib import Path

try:
    file_path = Path(__file__).resolve()
except NameError:
    file_path = Path.cwd()

shapefile_path = file_path.parent.parent.parent / "data/Counties_and_Unitary_Authorities_December_2023_Boundaries"
assert shapefile_path.is_dir()


In [3]:


def load_shapefile(path: Union[str, Path]) -> gpd.GeoDataFrame:
    """Load a shapefile or directory containing shapefile components.

    Accepts either a path to a `.shp` file or a directory containing shapefile
    files. Returns a GeoDataFrame.
    """
    p = Path(path)

    if p.is_dir():
        # try to find a .shp file in the directory
        shp_files = list(p.glob("*.shp"))
        if not shp_files:
            raise FileNotFoundError(f"No .shp files found in directory: {p}")
        shp_path = shp_files[0]
    else:
        shp_path = p

    if not shp_path.exists():
        raise FileNotFoundError(f"Shapefile not found: {shp_path}")

    # Read with geopandas; let geopandas handle encodings/CRS
    gdf = gpd.read_file(str(shp_path))

    return gdf

In [4]:
gdf = load_shapefile(path=shapefile_path)

In [5]:
devon_gdf = gdf[gdf["CTYUA23NM"] == "Devon"]

In [6]:
# Visualise the loaded shapefile on an interactive folium map
gdf = gdf.to_crs(epsg=4326)

# Compute a sensible center for the map
geom_name = devon_gdf.geometry.name
centroid = devon_gdf.geometry.unary_union.centroid
map_center = [centroid.y, centroid.x]

tooltip_fields = [c for c in gdf.columns if c != geom_name][:5]
tooltip = folium.GeoJsonTooltip(fields=tooltip_fields, aliases=tooltip_fields) if tooltip_fields else None

m = folium.Map(
    location=map_center, 
    zoom_start=9,
    width="100%",
    height="500px",
    scrollWheelZoom=False,
    tiles="CartoDB Positron")

folium.GeoJson(
    devon_gdf.__geo_interface__,
    name="shapefile",
    style_function=lambda feat: {
        "fillColor": "#ff7800",
        "color": "black",
        "weight": 0.5,
        "fillOpacity": 0.4,
    },
).add_to(m)

folium.LayerControl().add_to(m)

# Display the map
m

/tmp/ipykernel_167023/3949944462.py:6: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  centroid = devon_gdf.geometry.unary_union.centroid


In [7]:
from eoflow.ea import get_ea_water_quality
get_ea_water_quality?

Signature:
get_ea_water_quality(
    determinand: str,
    start_date: str,
    end_date: str,
    area: Optional[str] = None,
    polygon: Union[Dict, str, NoneType] = None,
    verbose: bool = True,
) -> pandas.core.frame.DataFrame
Docstring:
Convenience function to fetch EA water quality data with minimal setup.

Uses configuration values from the config file for API settings.

Args:
    determinand: Determinand code (e.g., "0076")
    start_date: Start date in YYYY-MM-DD format
    end_date: End date in YYYY-MM-DD format
    area: Precanned area code (e.g., "environment_agency,DCS")
    polygon: GeoJSON polygon for spatial filtering
    verbose: Print progress messages

Returns:
    DataFrame with water quality data

Example:
    >>> # Get temperature data for a specific EA area
    >>> df = get_ea_water_quality(
    ...     determinand="0076",
    ...     start_date="2024-01-01",
    ...     end_date="2024-12-31",
    ...     area="environment_agency,DCS"
    ... )
File:      ~/Wo

In [ ]:
wq_data = get_ea_water_quality(
    determinand="0076",
    start_date="2024-01-01",
    end_date="2024-12-31",
    polygon=devon_gdf)